<a href="https://colab.research.google.com/github/timfan705/CAmarket/blob/main/frontend_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CAmarket Frontend (Colab)

Run the Streamlit frontend in Colab and open it with a public URL.


## Quick Start
1. Run install cell.
2. Upload `main.py` when prompted.
3. Start Streamlit.
4. Run Cloudflare URL cell and open the link.


In [ ]:
!pip -q install \
  "streamlit>=1.32" \
  "pandas>=2.2" \
  "numpy>=1.26" \
  "scikit-learn>=1.4" \
  "folium>=0.16" \
  "streamlit-folium>=0.20" \
  "plotly>=5.20"

import sys
print('Python:', sys.version.split()[0])


## 1) Upload app file
If `/content/main.py` is missing, upload it from your local project.


In [ ]:
from pathlib import Path
APP_PATH = Path('/content/main.py')

if not APP_PATH.exists():
    from google.colab import files
    print('Upload main.py')
    uploaded = files.upload()
    if 'main.py' not in uploaded:
        raise FileNotFoundError('Upload must include main.py')

print('Using:', APP_PATH)
!wc -l /content/main.py


## 2) Start Streamlit


In [ ]:
import socket
import subprocess
import sys
import time

subprocess.run("pkill -f 'streamlit run' || true", shell=True)

LOG_PATH = '/content/streamlit.log'
log_file = open(LOG_PATH, 'w')
proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run', '/content/main.py',
        '--server.port', '8501',
        '--server.address', '0.0.0.0',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

for _ in range(20):
    s = socket.socket()
    try:
        s.connect(('127.0.0.1', 8501))
        s.close()
        break
    except Exception:
        time.sleep(1)

print('Streamlit running. Log:', LOG_PATH)


## 3) Open public URL
Run this cell and open the printed `trycloudflare.com` link.


In [ ]:
import re
import time
import subprocess

subprocess.run("pkill -f cloudflared || true", shell=True)
subprocess.run("rm -f /content/cloudflared.log", shell=True)
subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", shell=True, check=True)
subprocess.run("chmod +x cloudflared", shell=True, check=True)

subprocess.Popen(
    "./cloudflared tunnel --url http://127.0.0.1:8501 --no-autoupdate --loglevel info > /content/cloudflared.log 2>&1",
    shell=True,
)

public_url = None
for _ in range(60):
    txt = subprocess.check_output("cat /content/cloudflared.log || true", shell=True, text=True)
    m = re.search(r"https://[-a-z0-9]+\\.trycloudflare\\.com", txt)
    if m:
        public_url = m.group(0)
        break
    time.sleep(1)

if public_url:
    print('Open this URL:', public_url)
else:
    print('URL not ready. Run diagnostics cell below.')


## Optional: Diagnostics
Use only if the URL does not load.


In [ ]:
!curl -I http://127.0.0.1:8501
!tail -n 120 /content/streamlit.log
!tail -n 120 /content/cloudflared.log
